# Functions, Methods & Error Handling

This notebook covers:
1. Error handling fundamentals (LBYL vs EAFP)
2. Functions - bad patterns vs good patterns
3. Classes - bad patterns vs good patterns

## Part 1: Error Handling

Python has two philosophies for error handling. Understanding when to use each is crucial.

### LBYL - Look Before You Leap

Check if something is possible BEFORE you try it.

In [ ]:
# LBYL approach - check first, then act
users = {"alice": {"email": "alice@example.com", "age": 30}}

username = "bob"

if username in users:
    email = users[username]["email"]
    print(f"Email: {email}")
else:
    print(f"User {username} not found")

In [ ]:
# Another LBYL example - checking before converting
user_input = "not_a_number"

if user_input.isdigit():
    number = int(user_input)
    print(f"Number: {number}")
else:
    print("Invalid input - not a number")

### EAFP - Easier to Ask for Forgiveness than Permission

Just try it. If it fails, handle the error.

In [ ]:
# EAFP approach - try it, catch errors
users = {"alice": {"email": "alice@example.com", "age": 30}}

username = "bob"

try:
    email = users[username]["email"]
    print(f"Email: {email}")
except KeyError:
    print(f"User {username} not found")

In [ ]:
# EAFP with type conversion
user_input = "not_a_number"

try:
    number = int(user_input)
    print(f"Number: {number}")
except ValueError:
    print("Invalid input - not a number")

### When to use which?

**LBYL is better when:**
- The check is simple and cheap
- You want to avoid exceptions for flow control
- The condition is unlikely to change between check and use

**EAFP is better when:**
- The operation might fail for multiple reasons
- Race conditions are possible (file might be deleted between check and use)
- The check would be more complex than just trying
- You're working with external resources (files, network, databases)

In [ ]:
# EAFP is better for files - race condition possible with LBYL
import os

# BAD (LBYL) - file could be deleted between check and open
# if os.path.exists("config.json"):
#     with open("config.json") as f:
#         config = f.read()

# GOOD (EAFP) - handles the actual failure
try:
    with open("config.json") as f:
        config = f.read()
except FileNotFoundError:
    print("Config file not found, using defaults")
    config = "{}"

### raise - Signaling errors to the caller

When something goes wrong in a function, you have choices:
1. Return `None` or `False` - only for simple binary situations
2. `raise` an exception - let the caller decide what to do

`raise` makes functions more reusable because the caller controls error handling.

In [ ]:
# BAD - returning None doesn't tell you WHAT went wrong
def get_user_bad(users: dict, username: str):
    if username in users:
        return users[username]
    return None  # Was the user not found? Was there an error? Who knows.

result = get_user_bad({}, "alice")
if result is None:
    print("Something went wrong... but what?")

In [ ]:
# GOOD - raising tells you exactly what went wrong
def get_user_good(users: dict, username: str) -> dict:
    if username not in users:
        raise KeyError(f"User '{username}' not found")
    return users[username]

try:
    result = get_user_good({}, "alice")
except KeyError as e:
    print(f"Error: {e}")

### Custom Exceptions

Custom exceptions make it clearer what went wrong and allow callers to catch specific errors.

In [ ]:
# Define custom exceptions
class UserNotFoundError(Exception):
    """Raised when a user is not found in the system."""
    pass

class InvalidEmailError(Exception):
    """Raised when an email format is invalid."""
    pass

class InsufficientFundsError(Exception):
    """Raised when account balance is too low."""
    pass

In [ ]:
# Using custom exceptions
def get_user(users: dict, username: str) -> dict:
    if username not in users:
        raise UserNotFoundError(f"User '{username}' does not exist")
    return users[username]


def validate_email(email: str) -> str:
    if "@" not in email or "." not in email:
        raise InvalidEmailError(f"Invalid email format: {email}")
    return email


# Now callers can catch specific errors
try:
    user = get_user({}, "alice")
except UserNotFoundError as e:
    print(f"User error: {e}")
except InvalidEmailError as e:
    print(f"Email error: {e}")

### Why exceptions? The problem with returning False/None

When a function can fail in multiple ways, returning `False` or `None` doesn't tell you WHAT went wrong.

In [ ]:
# BAD: Returning False/None - can't distinguish errors

def register_user_bad(username: str, email: str, age: int):
    """
    Returns False if anything goes wrong.
    But WHAT went wrong? We can't tell!
    """
    # Multiple things can fail...
    if not username or len(username) < 3:
        return False  # Username too short
    
    if "@" not in email:
        return False  # Invalid email
    
    if age < 18:
        return False  # Too young
    
    return {"username": username, "email": email, "age": age}


# The problem: we can't tell what failed
result = register_user_bad("ab", "bad-email", 15)

if result is False:
    # Was it the username? The email? The age? ALL of them?
    # We have no idea! We can only show a generic error.
    print("Registration failed - but why?")
else:
    print(f"User registered: {result}")

In [ ]:
# GOOD: Different exceptions for different errors

class UsernameTooShortError(Exception):
    pass

class InvalidEmailError(Exception):
    pass

class UserTooYoungError(Exception):
    pass


def register_user_good(username: str, email: str, age: int) -> dict:
    """
    Raises specific exceptions for each type of error.
    The caller knows exactly what went wrong.
    """
    if not username or len(username) < 3:
        raise UsernameTooShortError(f"Username must be at least 3 characters, got '{username}'")
    
    if "@" not in email:
        raise InvalidEmailError(f"Email must contain @, got '{email}'")
    
    if age < 18:
        raise UserTooYoungError(f"Must be 18 or older, got {age}")
    
    return {"username": username, "email": email, "age": age}


# Now we can handle each error specifically
try:
    user = register_user_good("ab", "bad-email", 15)
    print(f"User registered: {user}")
except UsernameTooShortError as e:
    print(f"Username problem: {e}")
except InvalidEmailError as e:
    print(f"Email problem: {e}")
except UserTooYoungError as e:
    print(f"Age problem: {e}")

### Don't raise the same exception for different errors

If you raise `ValueError` for multiple different problems, the caller can't distinguish them.

In [ ]:
# BAD: Same exception type for different errors

def process_payment_bad(amount: float, card_number: str) -> bool:
    if amount <= 0:
        raise ValueError("Invalid amount")  # ValueError for amount
    
    if len(card_number) != 16:
        raise ValueError("Invalid card")  # ValueError for card too!
    
    return True


# Problem: we can only catch ValueError, but we don't know which one
try:
    process_payment_bad(-50, "123")
except ValueError as e:
    # Was it the amount or the card? We have to parse the message string!
    print(f"Error: {e}")
    # This is fragile - what if the message changes?

In [ ]:
# GOOD: Different exception types for different errors

class InvalidAmountError(Exception):
    pass

class InvalidCardError(Exception):
    pass


def process_payment_good(amount: float, card_number: str) -> bool:
    if amount <= 0:
        raise InvalidAmountError(f"Amount must be positive, got {amount}")
    
    if len(card_number) != 16:
        raise InvalidCardError(f"Card number must be 16 digits, got {len(card_number)}")
    
    return True


# Now we can handle each error type differently
try:
    process_payment_good(-50, "123")
except InvalidAmountError as e:
    print(f"Fix the amount: {e}")
except InvalidCardError as e:
    print(f"Fix the card: {e}")

### When to handle vs let it bubble up?

Sometimes the right answer is to NOT catch an exception.

**Catch when:**
- You can actually DO something about the error
- You need to translate it to a different error
- You're at the boundary of your application (API endpoint, CLI, etc.)

**Let it bubble up when:**
- You can't meaningfully handle it
- A higher-level caller should decide what to do
- It's a programming error (let it crash loudly)

In [ ]:
# Example: let errors bubble up to the right level

def calculate_average(numbers: list[float]) -> float:
    """Low-level function - raises, doesn't catch."""
    if not numbers:
        raise ValueError("Cannot calculate average of empty list")
    return sum(numbers) / len(numbers)


def get_student_average(student_id: int, grades_db: dict) -> float:
    """Mid-level function - lets ValueError bubble up."""
    grades = grades_db.get(student_id, [])
    return calculate_average(grades)  # Don't catch here - we can't handle it


def display_student_report(student_id: int, grades_db: dict) -> str:
    """High-level function - this is where we handle the error."""
    try:
        avg = get_student_average(student_id, grades_db)
        return f"Student {student_id}: Average grade {avg:.1f}"
    except ValueError:
        return f"Student {student_id}: No grades recorded"


# Test it
grades_db = {
    1: [85, 90, 78],
    2: [],  # No grades
}

print(display_student_report(1, grades_db))
print(display_student_report(2, grades_db))
print(display_student_report(3, grades_db))  # Student doesn't exist

## Part 2: Functions - Bad vs Good

Now let's look at a complete example. We'll build a simple order processing system.

First, we'll see how NOT to write it. Then we'll fix it.

### The BAD Version

This code works, but it has many problems. Can you spot them?

In [ ]:
# BAD VERSION - Order Processing System
# This code has MANY problems. Don't write code like this!

orders = []  # Global variable - BAD
total_revenue = 0  # Global variable - BAD

def process_order(customer, items, payment):
    global total_revenue  # Using global keyword - FORBIDDEN
    
    # No type annotations - hard to understand what this function expects
    
    # Doing too many things in one function:
    # 1. Validating input
    # 2. Calculating total
    # 3. Processing payment
    # 4. Updating global state
    # 5. Printing output
    
    # Validation mixed with logic
    if customer == "" or customer == None:
        print("Error: no customer")
        return
    
    if len(items) == 0:
        print("Error: no items")
        return
    
    # Calculate total (buried in the middle of other logic)
    total = 0
    for item in items:
        total = total + item["price"] * item["quantity"]
    
    # Apply discount (magic numbers, no explanation)
    if total > 100:
        total = total * 0.9
    
    # Check payment (validation mixed with processing)
    if payment < total:
        print("Error: not enough money")
        return
    
    # Update global state - BAD
    total_revenue = total_revenue + total
    
    # Create order and add to global list - BAD
    order = {"customer": customer, "items": items, "total": total}
    orders.append(order)
    
    # Function prints instead of returning - BAD
    print(f"Order processed for {customer}")
    print(f"Total: ${total}")
    print(f"Change: ${payment - total}")

In [ ]:
# Using the bad version
items = [
    {"name": "Coffee", "price": 5.0, "quantity": 2},
    {"name": "Sandwich", "price": 12.0, "quantity": 1}
]

process_order("Alice", items, 50.0)

# Problems:
# - We can't get the order back (it just prints)
# - We can't test this easily
# - We can't reuse parts of this logic
# - Global state makes it unpredictable

### What's wrong with the bad version?

1. **Global variables** - `orders` and `total_revenue` are global. Any function can change them. Hard to track, hard to test.

2. **`global` keyword** - Modifying global state from inside a function. Never do this.

3. **No type annotations** - What is `customer`? A string? A dict? What is `items`? Impossible to know without reading the code.

4. **Function does too much** - Validates, calculates, processes payment, updates state, AND prints. Should be 4-5 separate functions.

5. **Prints instead of returns** - We can't use the result. We can't test it. We can't build on it.

6. **Errors are just printed** - We can't catch them. We can't react to them programmatically.

7. **Magic numbers** - What is `0.9`? What is `100`? No explanation.

### The GOOD Version

Same functionality, but properly structured.

In [ ]:
# GOOD VERSION - Order Processing System

# Custom exceptions - clear error signaling
class OrderError(Exception):
    """Base exception for order processing errors."""
    pass

class EmptyOrderError(OrderError):
    """Raised when an order has no items."""
    pass

class InsufficientPaymentError(OrderError):
    """Raised when payment doesn't cover the total."""
    pass

class InvalidCustomerError(OrderError):
    """Raised when customer name is invalid."""
    pass

In [ ]:
# Single-responsibility functions

def validate_customer(customer: str) -> str:
    """Validate customer name. Raises InvalidCustomerError if invalid."""
    if not customer or not customer.strip():
        raise InvalidCustomerError("Customer name cannot be empty")
    return customer.strip()


def validate_items(items: list[OrderItem]) -> list[OrderItem]:
    """Validate order items. Raises EmptyOrderError if no items."""
    if not items:
        raise EmptyOrderError("Order must contain at least one item")
    return items


def calculate_subtotal(items: list[OrderItem]) -> float:
    """Calculate subtotal from items. Pure function - no side effects."""
    return sum(item["price"] * item["quantity"] for item in items)


def calculate_discount(subtotal: float) -> float:
    """Calculate discount amount based on subtotal."""
    if subtotal >= DISCOUNT_THRESHOLD:
        return subtotal * DISCOUNT_RATE
    return 0.0


def validate_payment(payment: float, total: float) -> float:
    """Validate payment covers total. Raises InsufficientPaymentError if not."""
    if payment < total:
        raise InsufficientPaymentError(
            f"Payment ${payment:.2f} is less than total ${total:.2f}"
        )
    return payment - total  # Return change

In [ ]:
# Main function - composes the smaller functions
def create_order(customer: str, items: list[OrderItem]) -> Order:
    """
    Create an order from customer and items.
    
    Raises:
        InvalidCustomerError: If customer name is invalid
        EmptyOrderError: If no items provided
    """
    validated_customer = validate_customer(customer)
    validated_items = validate_items(items)
    
    subtotal = calculate_subtotal(validated_items)
    discount = calculate_discount(subtotal)
    total = subtotal - discount
    
    return {
        "customer": validated_customer,
        "items": validated_items,
        "subtotal": subtotal,
        "discount": discount,
        "total": total
    }


def process_payment(order: Order, payment: float) -> dict:
    """
    Process payment for an order.
    
    Raises:
        InsufficientPaymentError: If payment doesn't cover total
    """
    change = validate_payment(payment, order["total"])
    
    return {
        "order": order,
        "payment": payment,
        "change": change,
        "success": True
    }

In [ ]:
# Using the GOOD version
items: list[OrderItem] = [
    {"name": "Coffee", "price": 5.0, "quantity": 2},
    {"name": "Sandwich", "price": 12.0, "quantity": 1}
]

try:
    order = create_order("Alice", items)
    print(f"Order created: {order}")
    
    result = process_payment(order, 50.0)
    print(f"Payment successful! Change: ${result['change']:.2f}")
    
except InvalidCustomerError as e:
    print(f"Customer error: {e}")
except EmptyOrderError as e:
    print(f"Order error: {e}")
except InsufficientPaymentError as e:
    print(f"Payment error: {e}")

In [ ]:
# Error handling in action - each error is specific and catchable

# Empty customer
try:
    order = create_order("", items)
except InvalidCustomerError as e:
    print(f"Caught: {e}")

# Empty order
try:
    order = create_order("Bob", [])
except EmptyOrderError as e:
    print(f"Caught: {e}")

# Insufficient payment
try:
    order = create_order("Charlie", items)
    result = process_payment(order, 5.0)  # Not enough!
except InsufficientPaymentError as e:
    print(f"Caught: {e}")

### Why is the GOOD version better?

| Bad Version | Good Version |
|-------------|--------------|
| Global variables | No global state |
| `global` keyword | Functions return values |
| No type hints | Full type annotations |
| One massive function | Small, focused functions |
| Prints output | Returns data |
| Errors are printed | Errors are raised |
| Magic numbers | Named constants |
| Can't test easily | Each function is testable |
| Can't reuse parts | Functions are composable |

## Part 3: Classes - Bad vs Good

The same principles apply to classes. Let's build a simple bank account.

### The BAD Class

In [ ]:
# BAD CLASS - Bank Account
# Same problems as the bad functions

all_accounts = []  # Global list - BAD

class BadBankAccount:
    def __init__(self, owner, balance):
        # No type hints
        self.owner = owner
        self.balance = balance
        all_accounts.append(self)  # Modifying global state - BAD
    
    def withdraw(self, amount):
        # No validation
        # No error signaling
        # Just prints
        if amount > self.balance:
            print("Not enough money")
            return
        self.balance = self.balance - amount
        print(f"Withdrew {amount}, new balance: {self.balance}")
    
    def deposit(self, amount):
        # No validation for negative amounts
        self.balance = self.balance + amount
        print(f"Deposited {amount}, new balance: {self.balance}")
    
    def transfer(self, other, amount):
        # Method does too much
        # No error handling
        if amount > self.balance:
            print("Not enough money")
            return
        self.balance = self.balance - amount
        other.balance = other.balance + amount
        print(f"Transferred {amount} to {other.owner}")

### The GOOD Class

In [ ]:
# GOOD CLASS - Bank Account

class InsufficientFundsError(Exception):
    """Raised when account balance is too low for a transaction."""
    pass

class InvalidAmountError(Exception):
    """Raised when transaction amount is invalid."""
    pass


class BankAccount:
    """A bank account with proper error handling and type hints."""
    
    def __init__(self, owner: str, initial_balance: float = 0.0):
        if not owner or not owner.strip():
            raise ValueError("Owner name cannot be empty")
        if initial_balance < 0:
            raise ValueError("Initial balance cannot be negative")
        
        self._owner = owner.strip()
        self._balance = initial_balance
    
    @property
    def owner(self) -> str:
        return self._owner
    
    @property
    def balance(self) -> float:
        return self._balance
    
    def _validate_amount(self, amount: float) -> None:
        """Validate transaction amount. Raises InvalidAmountError."""
        if amount <= 0:
            raise InvalidAmountError(f"Amount must be positive, got {amount}")
    
    def deposit(self, amount: float) -> float:
        """
        Deposit money into account.
        
        Returns:
            New balance after deposit
            
        Raises:
            InvalidAmountError: If amount is not positive
        """
        self._validate_amount(amount)
        self._balance += amount
        return self._balance
    
    def withdraw(self, amount: float) -> float:
        """
        Withdraw money from account.
        
        Returns:
            New balance after withdrawal
            
        Raises:
            InvalidAmountError: If amount is not positive
            InsufficientFundsError: If balance is too low
        """
        self._validate_amount(amount)
        if amount > self._balance:
            raise InsufficientFundsError(
                f"Cannot withdraw ${amount:.2f}, balance is ${self._balance:.2f}"
            )
        self._balance -= amount
        return self._balance
    
    def __repr__(self) -> str:
        return f"BankAccount(owner='{self._owner}', balance={self._balance:.2f})"

In [ ]:
# Using the GOOD class

account = BankAccount("Alice", 100.0)
print(account)

# Deposit
new_balance = account.deposit(50.0)
print(f"After deposit: ${new_balance:.2f}")

# Withdraw
new_balance = account.withdraw(30.0)
print(f"After withdrawal: ${new_balance:.2f}")

# Error handling
try:
    account.withdraw(500.0)  # Too much!
except InsufficientFundsError as e:
    print(f"Error: {e}")

try:
    account.deposit(-10.0)  # Invalid amount!
except InvalidAmountError as e:
    print(f"Error: {e}")

## Summary

### Error Handling
- **LBYL** - Check before you act. Good for simple checks.
- **EAFP** - Try and catch. Good for external resources, complex checks.
- **raise** - Signal errors to the caller. Makes code reusable.
- **Custom exceptions** - Make errors clear and catchable.
- **Let errors bubble up** - When you can't handle them meaningfully.

### Functions
- **Single responsibility** - One function, one job
- **No global state** - Never use the `global` keyword
- **Type annotations** - Make your code explicit
- **Return values** - Don't just print
- **Raise exceptions** - Let the caller decide how to handle errors
- **Named constants** - No magic numbers

### Classes
- Same principles apply
- Use `_underscore` for internal attributes
- Properties for controlled access
- Validate in `__init__`
- Methods should raise, not print

In [ ]:
# Single-responsibility functions

def validate_customer(customer: str) -> str:
    \"\"\"Validate customer name. Raises InvalidCustomerError if invalid.\"\"\"\n    if not customer or not customer.strip():\n        raise InvalidCustomerError(\"Customer name cannot be empty\")\n    return customer.strip()\n\n\ndef validate_items(items: list[OrderItem]) -> list[OrderItem]:\n    \"\"\"Validate order items. Raises EmptyOrderError if no items.\"\"\"\n    if not items:\n        raise EmptyOrderError(\"Order must contain at least one item\")\n    return items\n\n\ndef calculate_subtotal(items: list[OrderItem]) -> float:\n    \"\"\"Calculate subtotal from items. Pure function - no side effects.\"\"\"\n    return sum(item[\"price\"] * item[\"quantity\"] for item in items)\n\n\ndef calculate_discount(subtotal: float) -> float:\n    \"\"\"Calculate discount amount based on subtotal.\"\"\"\n    if subtotal >= DISCOUNT_THRESHOLD:\n        return subtotal * DISCOUNT_RATE\n    return 0.0\n\n\ndef validate_payment(payment: float, total: float) -> float:\n    \"\"\"Validate payment covers total. Raises InsufficientPaymentError if not.\"\"\"\n    if payment < total:\n        raise InsufficientPaymentError(\n            f\"Payment ${payment:.2f} is less than total ${total:.2f}\"\n        )\n    return payment - total  # Return change